In [1]:
# Processing csv files from the GBD for country specific BMR to xarray format
# Run for each health variable

In [9]:
import os
import xarray as xr
import numpy as np
import pandas as pd

In [3]:
# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS, LUNG_CANCER, STROKE
# resp_copd, t2_dm, cvd_ihd, lri, neo_lung, cvd_stroke
health_vars_out = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
                   "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [4]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"

# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS, LUNG_CANCER, STROKE
# Abbreviations below from vizhub download (in order to match the above)
health_vars_in = ["COPD", "DM2", "IHD", "LRI", "LC", "Stroke"]

In [14]:
for i in range(len(health_vars_in)):
    print(f"Processing health variable {health_vars_out[i]}")
    health_VAR = health_vars_in[i]
    health_VAR_out = health_vars_out[i]

    # Reads the CSV file into a DataFrame
    # Chronic Obstructive Pulmonary Disease baseline mortality rate by country
    df = pd.read_csv(f"{BMR_DIR}IHME-GBD_2021_DATA-{health_VAR}.csv")

    # Options are Number, Percent or Rate (Rate is per 100,000)
    # We will need to divide by 100,000 later to get rate per 1
    df = df[df["metric_name"] == "Rate"]

    # Calculate the mean across 1990-2009 for each country
    # We use the 1990-2009 mean for the BMR in the year 2000 and in all future
    # projections
    df_mean = df.groupby("location_name").mean("year").reset_index()

    country = df_mean["location_name"]
    val = df_mean["val"]  # the mean value [GBD Results Tool User Guide]
    upper = df_mean["upper"]  # 95% Confidence Interval Upper Bound
    lower = df_mean["lower"]  # 95% Confidence Interval Lower Bound

    data = np.stack([lower, val, upper], axis=1)  # shape (204 countries, 3 stats)
    
    # Create the xarray DataArray
    da = xr.DataArray(
        data,
        dims=["country", "quantile"],
        coords={
            "country": country,
            "quantile": ["lower", "mean", "upper"]
        },
        name="BMR_by_country"
    )

    # Dividing to find rate per 1 person
    # BMR from GBD (https://vizhub.healthdata.org/gbd-results/) is provided as a
    # RATE per 100,000. i.e if the rate was 10 per 100,000 the value provided would
    # be 10, not 0.0001
    # To calculate the mortality we must divide the rate by 100,000 to convert to
    # a per-person basis.
    da = da / 100000
    
    cite = ("Global Burden of Disease Collaborative Network. Global Burden of"
            "Disease Study 2021 (GBD 2021) Results. Seattle, United States: "
            "Institute for Health Metrics and Evaluation (IHME), 2022. Available "
            "from https://vizhub.healthdata.org/gbd-results/.")
    
    da.attrs["description"] = ("Average Baseline Mortality Rate per person per country for "
                               f"{health_VAR_out} from 1990-2009")
    da.attrs["citation"] = cite

    out_file = f"GBD_BMR_Country_{health_VAR_out}_1990-2009.nc"
    out_path = os.path.join(BMR_DIR, out_file)
    da.to_netcdf(out_path)

print("All processing complete.")

Processing health variable COPD
Processing health variable DIABETES
Processing health variable ISCHEMIC_HEART_DISEASE
Processing health variable LOWER_RESPIRATORY_INFECTIONS
Processing health variable LUNG_CANCER
Processing health variable STROKE
All processing complete.
